# MMAct: separate R3D-18 Kinetics-400 video-only experiment

This is an isolated experiment. It does not replace or write to the existing nine-model benchmark outputs. It uses validation macro-F1 to select a checkpoint and evaluates the unchanged held-out test split once, after training. The trainer verifies that the existing subject split contains exactly 852 test sample IDs before starting.

Enable a Kaggle GPU and Internet access (needed for the official torchvision weight download). Stage the MMAct dataset at `data/genuine_mmact` using the repository ingestion workflow first.

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path('/kaggle/working/HAR-MMAct')
REPO_URL = 'https://github.com/CHENNAMSETTYVIKRAM/HAR-MMAct.git'
BRANCH = 'improved-models'
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'branch', '--show-current'], check=True)
os.chdir(PROJECT_ROOT) if 'os' in globals() else None
import os
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))


## R3D-18 experiment (50 epochs, initial batch size 4)

Automatic resume uses the latest checkpoint in the experiment-specific directory. If Kaggle runs out of GPU memory, rerun with a smaller `--batch_size` (for example 2); do not change the split or use test metrics to tune hyperparameters. All outputs are kept under `outputs/video_model_improvements/r3d18/`.

In [ ]:
!python scripts/train_r3d18.py \
    --data_root data/genuine_mmact \
    --video_backbone r3d18 \
    --mode video_only \
    --experiment_name r3d18_kinetics400_video_only \
    --epochs 50 \
    --batch_size 4 \
    --lr 0.0001 \
    --weight_decay 0.0001 \
    --dropout 0.35 \
    --seed 42 \
    --output_dir outputs/video_model_improvements/r3d18 \
    --cache_dir outputs/cache \
    --num_workers 2

In [ ]:
import json
manifest = json.loads(Path('outputs/video_model_improvements/r3d18/experiment_manifest.json').read_text())
metrics = json.loads(Path('outputs/video_model_improvements/r3d18/metrics.json').read_text())
print('completed epochs:', manifest['completed_epochs'])
print('test samples:', manifest['split_counts']['test'])
print('test metrics:', metrics['test_metrics'])